In [1]:
import os, shutil, json, hashlib
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
ART = PROJECT / "artifacts"
MODELS = PROJECT / "models"
MET = PROJECT / "metrics"
MODELS.mkdir(parents=True, exist_ok=True)
MET.mkdir(parents=True, exist_ok=True)

# Что копируем в models/ (модель + преобразования)
FILES_TO_COPY = [
    "lab2_span_tfidf.joblib",     # преобразование (TF-IDF)
    "lab2_span_logreg.joblib",    # модель (LogisticRegression)
]

copied = []
for name in FILES_TO_COPY:
    src = ART / name
    dst = MODELS / name
    if not src.exists():
        raise SystemExit(f"Нет файла {src} — выполните lab2_baseline_model.ipynb")
    shutil.copy2(src, dst)
    copied.append(dst)
    print(f"OK: {dst} ({dst.stat().st_size} байт)")

print(f"\nВсего скопировано: {len(copied)} файлов в {MODELS}")

OK: /home/mluser/mlproject/models/lab2_span_tfidf.joblib (351303 байт)
OK: /home/mluser/mlproject/models/lab2_span_logreg.joblib (2161615 байт)

Всего скопировано: 2 файлов в /home/mluser/mlproject/models


In [2]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
MODELS = PROJECT / "models"
MET = PROJECT / "metrics"
DATA_CHECKSUMS = PROJECT / "data" / "CHECKSUMS.sha256"

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. SHA-256 всех файлов модели
model_hashes = {}
for p in sorted(MODELS.glob("*.joblib")):
    model_hashes[p.name] = {
        "sha256": sha256(p),
        "size_bytes": p.stat().st_size,
    }
    print(f"{p.name:<30} {model_hashes[p.name]['sha256']}  "
          f"{model_hashes[p.name]['size_bytes']} байт")

# 2. SHA-256 файлов данных (из ЛР1/ЛР2)
data_hashes = {}
if DATA_CHECKSUMS.exists():
    for line in DATA_CHECKSUMS.read_text().splitlines():
        if not line.strip():
            continue
        h, name = line.split(maxsplit=1)
        data_hashes[name.strip()] = h
    print(f"\nДанные ({len(data_hashes)} файлов):")
    for name, h in data_hashes.items():
        print(f"  {name:<50} {h}")

# 3. Единый идентификатор данных (агрегированный хэш)
data_id_material = "|".join(f"{k}:{v}" for k, v in sorted(data_hashes.items()))
data_id = hashlib.sha256(data_id_material.encode()).hexdigest()
print(f"\ndata_id (агрегированный SHA-256): {data_id}")

# 4. Сохранить в metrics/model_artifacts.json
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "framework": "scikit-learn",
    "model_files": model_hashes,
    "data_files": data_hashes,
    "data_id": data_id,
    "data_source": "scanpatch/pii-ner-corpus-synthetic-controlled",
    "data_snapshot": "6af0587dde068d90d8a1080285159c374b46a09b",
}
out = MET / "model_artifacts.json"
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nСохранено: {out}")

# 5. Также models/SHA256SUMS — стандартный формат sha256sum
sums = MODELS / "SHA256SUMS"
with open(sums, "w", encoding="utf-8") as f:
    for name, info in model_hashes.items():
        f.write(f"{info['sha256']}  {name}\n")
print(f"Сохранено: {sums}")
print(sums.read_text())

lab2_span_logreg.joblib        98b80ba98fc606323673795bf35c14b73f4b5bb62b6d000e9bd54a4185f507ae  2161615 байт
lab2_span_tfidf.joblib         234d712c786767548b18dfba2182fc7ac96fa1bc564ce4524225c4b11994f566  351303 байт

Данные (3 файлов):
  pii-ner-corpus-synthetic-controlled-test.arrow     b83e61317f398e6c0cabbd2428c7e52251a7b688a197735128d507437c3e8e4b
  pii-ner-corpus-synthetic-controlled-train.arrow    2172e04c64f80b7451a500835eff8b0ba70820386669bc7727401a56669ed216
  dataset_info.json                                  eed5f519435f01b2a069d0aa9fceb9a4b8d05311a1836f23ccff6afa4fc99e56

data_id (агрегированный SHA-256): df2b9b24503d0ea8fee87bad4cf046169b77b78da353fb23f4140ebc581cb84b

Сохранено: /home/mluser/mlproject/metrics/model_artifacts.json
Сохранено: /home/mluser/mlproject/models/SHA256SUMS
98b80ba98fc606323673795bf35c14b73f4b5bb62b6d000e9bd54a4185f507ae  lab2_span_logreg.joblib
234d712c786767548b18dfba2182fc7ac96fa1bc564ce4524225c4b11994f566  lab2_span_tfidf.joblib



In [3]:
import os
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
GITIGNORE = PROJECT / ".gitignore"

# 4.1 .gitignore — минимальный, но покрывает Python/Jupyter/секреты/кэши
# ВАЖНО: models/*.joblib НЕ игнорируются — их нужно коммитить
GITIGNORE_CONTENT = """# Python
__pycache__/
*.py[cod]
*.pyo

# Virtual environments
venv/
.venv/
env/

# Jupyter
.ipynb_checkpoints/

# Test / cache
.pytest_cache/

# Secrets
.env
.env.*
*.pem
id_rsa

# IDE / OS
.vscode/
.idea/
.DS_Store

# Temporary / not-for-commit
*.tmp
*.log
"""
GITIGNORE.write_text(GITIGNORE_CONTENT, encoding="utf-8")
print(f"OK: {GITIGNORE}")
print(GITIGNORE.read_text())
# 4.2 requirements.txt — из активированного venv
import subprocess, sys

r = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                   capture_output=True, text=True)
lines = r.stdout.splitlines()
req_path = PROJECT / "requirements.txt"
req_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

KEY = ("scikit-learn", "pandas", "numpy", "pyarrow", "datasets",
       "joblib", "matplotlib", "pytest", "bandit", "jupyterlab")
print(f"OK: {req_path} ({len(lines)} пакетов)")
print("\nКлючевые пакеты:")
for l in lines:
    if any(l.lower().startswith(k) for k in KEY):
        print("  ", l)

OK: /home/mluser/mlproject/.gitignore
# Python
__pycache__/
*.py[cod]
*.pyo

# Virtual environments
venv/
.venv/
env/

# Jupyter
.ipynb_checkpoints/

# Test / cache
.pytest_cache/

# Secrets
.env
.env.*
*.pem
id_rsa

# IDE / OS
.vscode/
.idea/
.DS_Store

# Temporary / not-for-commit
*.tmp
*.log

OK: /home/mluser/mlproject/requirements.txt (155 пакетов)

Ключевые пакеты:
   bandit==1.9.4
   datasets==5.1.0
   joblib==1.6.0
   jupyterlab==4.6.4
   jupyterlab_pygments==0.3.0
   jupyterlab_server==2.28.1
   matplotlib==3.11.2
   matplotlib-inline==0.2.2
   numpy==2.4.6
   pandas==3.1.0
   pyarrow==26.0.0
   pytest==9.2.0
   scikit-learn==1.9.1


In [4]:
from pathlib import Path
import os

PROJECT = Path(os.path.expanduser("~/mlproject"))
README = PROJECT / "README.md"

lines = [
    "# ML Project: NLP для киберустойчивости (LR2)",
    "",
    "Обнаружение и классификация персональных данных (ПДн) в текстах",
    "согласно 152-ФЗ. Базовая модель — span-type classifier:",
    "TF-IDF (char_wb n-граммы) + LogisticRegression.",
    "",
    "## Структура проекта",
    "",
    "```",
    ".",
    "├── data/                   # датасет, CHECKSUMS.sha256",
    "├── notebooks/              # ноутбуки ЛР1/ЛР2",
    "├── src/                    # проверяемые модули",
    "│   ├── predictor.py        # валидация входа + предсказание",
    "│   ├── train.py            # обучение модели (CLI)",
    "│   └── __init__.py",
    "├── models/                 # финальные артефакты + SHA256SUMS",
    "├── artifacts/              # промежуточные артефакты",
    "├── metrics/                # метрики и отчёты",
    "├── tests/                  # pytest (11 негативных + 1 позитивный)",
    "├── requirements.txt",
    "├── .gitignore",
    "└── README.md",
    "```",
    "",
    "## Установка",
    "",
    "```bash",
    "python3 -m venv venv",
    "source venv/bin/activate",
    "pip install --upgrade pip",
    "pip install -r requirements.txt",
    "```",
    "",
    "## Данные",
    "",
    "Датасет: scanpatch/pii-ner-corpus-synthetic-controlled (Hugging Face).",
    "",
    "```bash",
    "sha256sum -c data/CHECKSUMS.sha256",
    "```",
    "",
    "## Обучение",
    "",
    "```bash",
    "source venv/bin/activate",
    "python -m src.train --artifacts artifacts --seed 42",
    "```",
    "",
    "Создаёт artifacts/lab2_span_tfidf.joblib, artifacts/lab2_span_logreg.joblib,",
    "artifacts/lab2_model_report.json.",
    "",
    "## Оценка на тесте",
    "",
    "```bash",
    "jupyter lab notebooks/lab2_metrics.ipynb",
    "cat metrics/baseline.json",
    "```",
    "",
    "Основная метрика — macro_f1 на отложенной тестовой выборке.",
    "",
    "## Тестирование (pytest)",
    "",
    "```bash",
    "source venv/bin/activate",
    "python -m pytest tests/ -q",
    "```",
    "",
    "Ожидается: 12 passed. Проверяются: неверный тип, NaN/Inf, пустой вход,",
    "лимиты, битый/подменённый файл. Для каждого негативного кейса — проверка,",
    "что модель не вызывается.",
    "",
    "## Предсказание",
    "",
    "```python",
    "import sys, os",
    "sys.path.insert(0, os.path.expanduser('~/mlproject'))",
    "from src.predictor import predict, predict_from_file",
    "",
    "result = predict(['ivan.petrov@example.com', '+7 900 123-45-67'])",
    "for r in result:",
    "    print(r['label'], r['confidence'], r['span'])",
    "",
    "result = predict_from_file('spans.txt')  # .txt, UTF-8, один спан на строку",
    "```",
    "",
    "## Статический анализ (Bandit)",
    "",
    "```bash",
    "source venv/bin/activate",
    "python -m bandit -r src/",
    "```",
    "",
    "Отчёт: metrics/bandit_report.json.",
    "",
    "## Артефакты модели",
    "",
    "Финальные артефакты — models/:",
    "- lab2_span_tfidf.joblib — TF-IDF-преобразование",
    "- lab2_span_logreg.joblib — LogisticRegression",
    "",
    "Проверка целостности:",
    "",
    "```bash",
    "cd models/",
    "sha256sum -c SHA256SUMS",
    "```",
    "",
    "Манифест с хэшами и data_id — metrics/model_artifacts.json.",
    "Параметры обучения — metrics/training_params.json.",
    "Метрики — metrics/metrics_summary.json, metrics/baseline.json.",
    "",
    "## Ограничения",
    "",
    "- Данные синтетические: высокая accuracy на test не гарантирует",
    "  обобщения на реальные ПДн.",
    "- Модель типизирует готовые спаны; извлечение спанов (полный NER) —",
    "  за рамками ЛР2.",
    "- .joblib использует pickle: загружать только собственные артефакты.",
    "- Jupyter запускать от mluser, слушать 127.0.0.1, доступ — через SSH-туннель.",
    "",
]

README_TEXT = "\n".join(lines)
README.write_text(README_TEXT, encoding="utf-8")

print(f"OK: {README} ({len(README_TEXT)} байт, {len(lines)} строк)")
print("\n--- первые 10 строк ---")
print("\n".join(README_TEXT.splitlines()[:10]))

OK: /home/mluser/mlproject/README.md (2950 байт, 121 строк)

--- первые 10 строк ---
# ML Project: NLP для киберустойчивости (LR2)

Обнаружение и классификация персональных данных (ПДн) в текстах
согласно 152-ФЗ. Базовая модель — span-type classifier:
TF-IDF (char_wb n-граммы) + LogisticRegression.

## Структура проекта

```
.


In [5]:
import os, sys, json, subprocess
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
MODELS = PROJECT / "models"

# 6.1 Проверка SHA256SUMS
r = subprocess.run(["sha256sum", "-c", "SHA256SUMS"],
                   cwd=str(MODELS), capture_output=True, text=True)
print("=== sha256sum -c SHA256SUMS ===")
print(r.stdout.strip())
print(f"returncode: {r.returncode}")

# 6.2 Загрузка финальной модели из models/ и предсказание
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import importlib
import src.predictor as pred_mod
importlib.reload(pred_mod)

pred_mod._DEFAULT_ART = MODELS
pred_mod._VECTORIZER = None
pred_mod._MODEL = None
pred_mod._EXPECTED_N_FEATURES = None

test_spans = [
    "ivan.petrov@example.com",
    "+7 900 123-45-67",
    "4276 1234 5678 9012",
    "123-456-789 00",
]
result = pred_mod.predict(test_spans)
print("\n=== Предсказания на финальной модели из models/ ===")
for r in result:
    print(f"  {r['label']:<20} conf={r['confidence']:.3f}  {r['span']!r}")

# 6.3 Проверка импорта src.train
print("\n=== Проверка src.train ===")
r = subprocess.run([sys.executable, "-c",
                    "import sys; sys.path.insert(0, %r); "
                    "from src.train import flatten_spans, train; "
                    "print('OK: src.train импортируется')" % str(PROJECT)],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

# 6.4 Итоговый манифест
manifest = json.loads((PROJECT / "metrics" / "model_artifacts.json").read_text())
print("\n=== ИТОГ ===")
print(f"data_id: {manifest['data_id']}")
for name, info in manifest["model_files"].items():
    print(f"{name:<30} {info['sha256']}")

=== sha256sum -c SHA256SUMS ===
lab2_span_logreg.joblib: OK
lab2_span_tfidf.joblib: OK
returncode: 0

=== Предсказания на финальной модели из models/ ===
  email                conf=0.921  'ivan.petrov@example.com'
  mobile_phone         conf=0.917  '+7 900 123-45-67'
  tin                  conf=0.326  '4276 1234 5678 9012'
  snils                conf=0.679  '123-456-789 00'

=== Проверка src.train ===
OK: src.train импортируется

=== ИТОГ ===
data_id: df2b9b24503d0ea8fee87bad4cf046169b77b78da353fb23f4140ebc581cb84b
lab2_span_logreg.joblib        98b80ba98fc606323673795bf35c14b73f4b5bb62b6d000e9bd54a4185f507ae
lab2_span_tfidf.joblib         234d712c786767548b18dfba2182fc7ac96fa1bc564ce4524225c4b11994f566
